My attempt at re-creating the algorithms talked about in the original research paper using the data in the sample_data folder as a benchmark (DW1000 data). I want to see if the issues I'm having are with my post-processing scripts, or with my registers from the DW3000 itself.

In [1]:
import numpy as np
import cmath
import math
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import pathlib
import re
from itertools import batched






useful jargon:
UWB: ultra-wide band
ToA: Time of Arrival
LoS: Line of sight
CIR: Channel Impulse Response
CSI: Channel state information



common variables:
d: distance
fc: carrier wave frequency
hlos: line of sight signal
alos: attenuation of los (attenuation is the loss of signal strength over a distance)
tao_los: time delay of los
Φ: phase of the UWB signal
lambda: wavelength of UWB signal (c / fc)
N: integer number of complete wrappings
CFO: carrier frequency offset
Φini: initial phase offset
delta_fc: frequency difference between transmitter and receiver
t: time
Φ_initiaitor_ini
Φ_responder_ini




### Equation 4:
```
error = 1/2*(clock_drift)*(time_of_flight)
```

---

### Equation 6:
```
tlos = distance / speed_of_light
hlos = attenuation * e ^ (-i * 2pi * carrier_frequency * tlos)
```
(hlos is a complex number)

---
### Equation 7:
collecting just the phase from the equation above
```
phase = arctan2(imaginary(hlos)/real(hlos))

phase = (2 * pi * carrier_frequency * distance / speed_of_light) % 2*pi
```
### Equation 8:
Explaining how equation 7 now wraps as a function of the wavelength (0-2pi)
```
distance_wrapped = (phase/2*pi * number_of_complete_wraps) * wavelength
```

### Equation 10:
re-writing how equation 6 and 7 should be with carrier frequency offset

```
phase(input_time) = (equation_7 - (2*pi * frequency_difference * input_time) + (initial_phase_of_transmitter - initial_p_of_receiver))
```

### Equation 11:
explaining how traditional distance taken with two-way ranging can resolve the ambiguity number, N

```
N = floor(distance_two_way_ranging / wavelength)
```

### Equation 12:
restating equation 10 with colors, where the transmitter is the initiator and the receiver is the responder

```
phase_of_poll(input_time) = ((-2 * pi * carrier_frequency * time_delay_of_los) - (2 * pi * delta_fc * input_time) + (inital_phase_offset_of_initiator - initial_phase_offset_of_responder)) % 2*pi
```

### Equation 13
explaining the phase of the response message

```
phase_of_response(input_time) = ((-2 * pi * (carrier_frequency + delta_fc) * time_delay_of_los) - (2*pi * delta_fc * input_time) - (inital_phase_offset_of_initiator - initial_phase_offset_of_responder)) % 2*pi
```
Also explains that delta_fc can be considered insignificant in calculations

### Equation 14
Says that initial offset and CFO can be cancelled by adding the two phases together

poll is gotten at 2,
response is sent at 3,
response is gotten at 4,

```
phase_poll + phase_response = -4*pi*carrier_frequency+2*pi*delta_fc*(time_at_response_tx + time_at_poll_rx) % 2*pi
```

---




I've refactored the code and the output seems stable (using fine-grained correction), but it still loves to jump from normal to inverted randomly.



